# Workshop 9: Reinforcement Learning

Welcome! In this notebook we'll go from the basics covered in the slides to actually implementing RL algorithms and watching them learn.

**What we'll build:**
1. **Multi-Armed Bandit** — the simplest RL problem (one state, multiple actions)
2. **Q-Learning on FrozenLake** — a classic gridworld from the Gymnasium library
3. **SARSA vs Q-Learning on CliffWalking** — see how on-policy vs off-policy leads to different behaviour

**Quick refresher from the slides:**
- The agent observes a **state**, picks an **action**, and receives a **reward**
- We want to learn a **policy** (what action to take in each state) that maximises total reward
- We estimate **Q(s,a)** — the value of taking action *a* in state *s* — and keep updating it as we gain experience

## Setup

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

np.random.seed(42)

---
## Part 1 — Multi-Armed Bandit (30 min)

### What is a bandit problem?

Imagine a row of $k$ slot machines ("one-armed bandits"). Each machine pays out a different average reward, but you don't know which is best. You have a limited number of pulls — how do you find the best machine quickly?

This is the **pure exploration vs exploitation** problem:
- **Exploit** — pull the machine you *think* is best right now
- **Explore** — try machines you haven't pulled much, in case a better one exists

There's only **one state** here, so we just need `Q[a]` (no state index).

In [ ]:
class BanditEnv:
    """k-armed bandit. Each arm has a hidden true value drawn from N(0,1).
    Calling step(action) returns a noisy reward around that true value."""

    def __init__(self, k=10, seed=None):
        rng = np.random.default_rng(seed)
        self.k = k
        self.true_values = rng.standard_normal(k)   # hidden — agent can't see these!
        self.best_action = np.argmax(self.true_values)

    def step(self, action):
        """Returns a noisy reward for the chosen arm."""
        return np.random.randn() + self.true_values[action]


env = BanditEnv(k=10, seed=0)
print('True values (hidden from agent):', np.round(env.true_values, 2))
print('Best arm:', env.best_action)

### 1.1 Implementing epsilon-greedy

The **epsilon-greedy** strategy:
- With probability `epsilon` → pick a **random** arm (explore)
- Otherwise → pick the arm with the **highest Q estimate** (exploit)

We update Q estimates using **exponential averaging**:
$$Q_{t+1}(a) = Q_t(a) + \alpha \left[r_{t+1} - Q_t(a)\right]$$

**Your task:** fill in the two `TODO` sections below.

In [ ]:
def run_bandit(env, n_steps=1000, epsilon=0.1, alpha=0.1):
    """
    Run epsilon-greedy on a BanditEnv for n_steps.
    Returns:
        rewards        - reward received at each step
        optimal_chosen - bool array: True when we picked the best arm
        Q              - final Q estimates
    """
    Q = np.zeros(env.k)
    rewards = np.zeros(n_steps)
    optimal_chosen = np.zeros(n_steps, dtype=bool)

    for t in range(n_steps):
        # --- TODO 1: Choose an action using epsilon-greedy ---
        # Hint: use np.random.random() < epsilon to decide whether to explore
        # Hint: use np.argmax(Q) to exploit
        action = None  # replace this


        # Get reward from environment
        reward = env.step(action)

        # --- TODO 2: Update Q[action] using exponential averaging ---
        # Q[action] = Q[action] + alpha * (reward - Q[action])


        rewards[t] = reward
        optimal_chosen[t] = (action == env.best_action)

    return rewards, optimal_chosen, Q


# Quick sanity check (won't plot yet)
rewards, optimal, Q_final = run_bandit(env, n_steps=500, epsilon=0.1)
print('Mean reward over last 100 steps:', np.mean(rewards[-100:]).round(2))
print('% optimal action in last 100 steps:', np.mean(optimal[-100:]) * 100, '%')

### 1.2 Comparing different epsilon values

Let's run the same bandit 20 times (different random seeds) for each epsilon and average the results to get a fair comparison.

In [ ]:
N_RUNS   = 20
N_STEPS  = 1000
EPSILONS = [0.0, 0.01, 0.1, 0.3]

results = {}  # epsilon -> average rewards over time

for eps in EPSILONS:
    all_rewards = np.zeros((N_RUNS, N_STEPS))
    for run in range(N_RUNS):
        e = BanditEnv(k=10, seed=run)
        r, _, _ = run_bandit(e, n_steps=N_STEPS, epsilon=eps, alpha=0.1)
        all_rewards[run] = r
    results[eps] = all_rewards.mean(axis=0)

plt.figure(figsize=(10, 4))
for eps, avg_rewards in results.items():
    plt.plot(avg_rewards, label=f'ε = {eps}')
plt.xlabel('Step')
plt.ylabel('Average reward')
plt.title('Epsilon-Greedy Bandit — Effect of Exploration Rate')
plt.legend()
plt.tight_layout()
plt.show()

**Discussion questions:**
- Which epsilon reaches the highest reward fastest? Why might that not always be the best choice?
- What happens with ε = 0 (pure greedy)? Why doesn't it improve over time?
- What's the downside of ε = 0.3?

### 1.3 Bonus: Decaying epsilon

A common trick is to start with high exploration and gradually reduce it — explore early, exploit late.

**Your task:** modify `run_bandit` (or write a new version) so that epsilon **decays** over time. One simple schedule: `epsilon_t = 1 / (t + 1)`. Plot it alongside the fixed-epsilon results above and see if it improves.

In [ ]:
# Your code here


---
## Part 2 — Q-Learning on FrozenLake (45 min)

### The environment

[FrozenLake-v1](https://gymnasium.farama.org/environments/toy_text/frozen_lake/) is a 4×4 grid. The agent starts at `S` and must reach the goal `G` without falling through holes `H`.

```
S F F F
F H F H
F F F H
H F F G
```
- **States:** 16 grid positions (0–15)
- **Actions:** Left (0), Down (1), Right (2), Up (3)
- **Reward:** +1 for reaching G, 0 everywhere else
- **Slippery:** by default the ice is slippery, so you don't always move the direction you intended!

We'll use `is_slippery=False` first to keep things simpler.

In [ ]:
env_fl = gym.make('FrozenLake-v1', is_slippery=False)

n_states  = env_fl.observation_space.n   # 16
n_actions = env_fl.action_space.n        # 4

print(f'States: {n_states},  Actions: {n_actions}')
print('Action meanings: 0=Left  1=Down  2=Right  3=Up')

### 2.1 Implementing Q-Learning

The Q-Learning update rule:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[r_t + \gamma \max_{a} Q(s_{t+1}, a) - Q(s_t, a_t)\right]$$

Note the `max` — we update towards the **best possible next action**, regardless of what we actually do next. That's what makes it **off-policy**.

**Your task:** fill in the `TODO` sections.

In [ ]:
def train_qlearning(env, n_episodes=5000, alpha=0.8, gamma=0.99,
                    epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.999):
    """
    Train a Q-table using Q-Learning with decaying epsilon-greedy exploration.
    Returns the learned Q-table and per-episode rewards.
    """
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    epsilon = epsilon_start
    episode_rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False

        while not done:
            # --- TODO 1: Epsilon-greedy action selection ---
            # Explore: env.action_space.sample()  /  Exploit: np.argmax(Q[state])
            action = None  # replace this


            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # --- TODO 2: Q-Learning update ---
            # target = reward + gamma * max(Q[next_state])   (use 'reward' alone if done)
            # Q[state, action] += alpha * (target - Q[state, action])


            total_reward += reward
            state = next_state

        # Decay epsilon
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(total_reward)

    return Q, episode_rewards


Q_table, ep_rewards = train_qlearning(env_fl, n_episodes=5000)
print('Training complete!')
print(f'Win rate in last 500 episodes: {np.mean(ep_rewards[-500:]) * 100:.1f}%')

### 2.2 Visualising the learned policy

Let's see what action the agent has learned to take in each cell.

In [ ]:
def plot_policy(Q, grid_size=4, title='Learned Policy'):
    """Display the greedy policy as arrows on the grid."""
    arrows = ['←', '↓', '→', '↑']
    holes  = {5, 7, 11, 12}  # FrozenLake-v1 default holes
    goal   = 15

    policy = np.argmax(Q, axis=1).reshape(grid_size, grid_size)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(0, grid_size)
    ax.set_ylim(0, grid_size)
    ax.set_xticks(range(grid_size + 1))
    ax.set_yticks(range(grid_size + 1))
    ax.grid(True)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13)

    for row in range(grid_size):
        for col in range(grid_size):
            s = row * grid_size + col
            cx, cy = col + 0.5, (grid_size - 1 - row) + 0.5
            if s in holes:
                ax.add_patch(plt.Rectangle((col, grid_size - 1 - row), 1, 1,
                                           color='#c0392b', alpha=0.6))
                ax.text(cx, cy, 'H', ha='center', va='center', fontsize=16, fontweight='bold', color='white')
            elif s == goal:
                ax.add_patch(plt.Rectangle((col, grid_size - 1 - row), 1, 1,
                                           color='#27ae60', alpha=0.6))
                ax.text(cx, cy, 'G', ha='center', va='center', fontsize=16, fontweight='bold', color='white')
            elif s == 0:
                ax.text(cx, cy, 'S', ha='center', va='center', fontsize=14, color='steelblue', fontweight='bold')
            else:
                ax.text(cx, cy, arrows[policy[row, col]],
                        ha='center', va='center', fontsize=20)

    plt.tight_layout()
    plt.show()


plot_policy(Q_table, title='Q-Learning — FrozenLake (not slippery)')

In [ ]:
# Plot the learning curve — smoothed over 100-episode windows
window = 100
smoothed = np.convolve(ep_rewards, np.ones(window) / window, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(smoothed)
plt.xlabel('Episode')
plt.ylabel(f'Avg reward (window={window})')
plt.title('Q-Learning on FrozenLake — Learning Curve')
plt.tight_layout()
plt.show()

### 2.3 Evaluate the trained policy

Now let's turn off exploration (epsilon = 0) and run 100 evaluation episodes to see how well the agent actually performs.

In [ ]:
def evaluate(Q, env, n_eval=100):
    wins = 0
    for _ in range(n_eval):
        state, _ = env.reset()
        done = False
        while not done:
            action = np.argmax(Q[state])          # always greedy
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
        wins += reward  # reward=1 only at goal
    return wins / n_eval


win_rate = evaluate(Q_table, env_fl)
print(f'Win rate (greedy, no exploration): {win_rate * 100:.0f}%')

### 2.4 Slippery ice (bonus)

Now try the harder version where actions are stochastic — the agent doesn't always move the direction it chooses. You may need more episodes and a lower alpha.

In [ ]:
env_slip = gym.make('FrozenLake-v1', is_slippery=True)

Q_slip, rewards_slip = train_qlearning(env_slip, n_episodes=20000,
                                       alpha=0.1, gamma=0.99,
                                       epsilon_start=1.0,
                                       epsilon_end=0.05,
                                       epsilon_decay=0.9995)

wr = evaluate(Q_slip, env_slip)
print(f'Win rate on slippery lake: {wr * 100:.0f}%')
plot_policy(Q_slip, title='Q-Learning — FrozenLake (slippery)')

**Discussion questions:**
- Why is the slippery version harder?
- What happens to the learned policy? Does it change compared to the non-slippery version?
- Why does the agent sometimes point *away* from the goal in the slippery version?

---
## Part 3 — SARSA vs Q-Learning on CliffWalking (15 min)

### The environment

[CliffWalking-v0](https://gymnasium.farama.org/environments/toy_text/cliff_walking/) is a 4×12 grid. The agent starts at the bottom-left and must reach the bottom-right. Stepping on the cliff (bottom row, except start and goal) sends you back to start with a -100 penalty.

```
o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
S  C  C  C  C  C  C  C  C  C  C  G
```

This is the classic example that highlights the **on-policy (SARSA) vs off-policy (Q-Learning)** difference:
- Q-Learning learns the shortest path right along the cliff edge (optimal, but risky)
- SARSA learns a safer path one row up (because it accounts for the fact that epsilon-greedy might accidentally step off the cliff during training)

In [ ]:
def train_sarsa(env, n_episodes=500, alpha=0.5, gamma=0.99, epsilon=0.1):
    """
    Train a Q-table using SARSA (on-policy TD control).
    Key difference from Q-learning: the target uses the ACTUAL next action
    chosen by the policy, not the max.
    """
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    episode_rewards = []

    def choose_action(state):
        if np.random.random() < epsilon:
            return env.action_space.sample()
        return np.argmax(Q[state])

    for episode in range(n_episodes):
        state, _ = env.reset()
        action = choose_action(state)  # choose BEFORE the loop
        total_reward = 0
        done = False

        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            next_action = choose_action(next_state)  # pick next action NOW

            # SARSA update: uses Q[next_state, next_action] — the actual next move
            if done:
                target = reward
            else:
                target = reward + gamma * Q[next_state, next_action]

            Q[state, action] += alpha * (target - Q[state, action])

            total_reward += reward
            state, action = next_state, next_action  # carry action forward!

        episode_rewards.append(total_reward)

    return Q, episode_rewards


def train_qlearning_cliff(env, n_episodes=500, alpha=0.5, gamma=0.99, epsilon=0.1):
    """Q-Learning (same as before, just fixed epsilon for fair comparison)."""
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    episode_rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False

        while not done:
            if np.random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(Q[state])

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            if done:
                target = reward
            else:
                target = reward + gamma * np.max(Q[next_state])  # uses max!

            Q[state, action] += alpha * (target - Q[state, action])

            total_reward += reward
            state = next_state

        episode_rewards.append(total_reward)

    return Q, episode_rewards


env_cw = gym.make('CliffWalking-v0')

Q_sarsa,  rewards_sarsa  = train_sarsa(env_cw,  n_episodes=500, epsilon=0.1)
Q_qlearn, rewards_qlearn = train_qlearning_cliff(env_cw, n_episodes=500, epsilon=0.1)

print('SARSA   — mean reward (last 50 episodes):', np.mean(rewards_sarsa[-50:]).round(1))
print('Q-Learn — mean reward (last 50 episodes):', np.mean(rewards_qlearn[-50:]).round(1))

In [ ]:
# Smooth and plot both learning curves together
window = 20

def smooth(x, w):
    return np.convolve(x, np.ones(w) / w, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(smooth(rewards_sarsa,  window), label='SARSA (on-policy)')
plt.plot(smooth(rewards_qlearn, window), label='Q-Learning (off-policy)')
plt.xlabel('Episode')
plt.ylabel(f'Avg reward (window={window})')
plt.title('SARSA vs Q-Learning on CliffWalking')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_cliff_policy(Q, title, grid_rows=4, grid_cols=12):
    """Visualise the greedy policy on the CliffWalking grid."""
    arrows   = ['↑', '→', '↓', '←']
    cliff    = set(range(37, 47))   # bottom row except start (36) and goal (47)
    start, goal = 36, 47

    policy = np.argmax(Q, axis=1).reshape(grid_rows, grid_cols)

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.set_xlim(0, grid_cols)
    ax.set_ylim(0, grid_rows)
    ax.set_xticks(range(grid_cols + 1))
    ax.set_yticks(range(grid_rows + 1))
    ax.grid(True)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13)

    for row in range(grid_rows):
        for col in range(grid_cols):
            s  = row * grid_cols + col
            cx = col + 0.5
            cy = (grid_rows - 1 - row) + 0.5
            if s in cliff:
                ax.add_patch(plt.Rectangle((col, grid_rows - 1 - row), 1, 1,
                                           color='#c0392b', alpha=0.6))
                ax.text(cx, cy, 'C', ha='center', va='center',
                        fontsize=9, color='white', fontweight='bold')
            elif s == goal:
                ax.add_patch(plt.Rectangle((col, grid_rows - 1 - row), 1, 1,
                                           color='#27ae60', alpha=0.6))
                ax.text(cx, cy, 'G', ha='center', va='center',
                        fontsize=12, color='white', fontweight='bold')
            elif s == start:
                ax.text(cx, cy, 'S', ha='center', va='center',
                        fontsize=12, color='steelblue', fontweight='bold')
            else:
                ax.text(cx, cy, arrows[policy[row, col]],
                        ha='center', va='center', fontsize=14)

    plt.tight_layout()
    plt.show()


plot_cliff_policy(Q_sarsa,  title='SARSA — Learned Policy')
plot_cliff_policy(Q_qlearn, title='Q-Learning — Learned Policy')

**What to look for:**
- **SARSA** should learn a route that hugs the *top* of the grid — safer because random exploration steps are less likely to send it off the cliff.
- **Q-Learning** should learn the optimal route *right along the cliff edge* — because it always imagines taking the best action, regardless of what exploration might accidentally do.

**Discussion:**
- Which algorithm gets higher reward *during training*? Which achieves a better *optimal* policy?
- In a real robotics setting, would you prefer SARSA or Q-Learning during training? Why?
- What would happen if you set epsilon to 0 during evaluation for both algorithms?

---
## Wrap-up

Today we implemented three classic RL algorithms from scratch:

| Algorithm | Type | Key update |
|-----------|------|------------|
| Epsilon-Greedy Bandit | Single-state RL | `Q += α(r - Q)` |
| Q-Learning | Off-policy TD | `Q += α(r + γ·max Q' - Q)` |
| SARSA | On-policy TD | `Q += α(r + γ·Q(s',a') - Q)` |

**Where to go from here:**
- **Deep Q-Networks (DQN):** Replace the Q-table with a neural network → can handle huge/continuous state spaces (e.g., learning from raw pixels)
- **Policy Gradient methods:** Instead of learning Q-values, directly learn the policy as a neural network
- **Actor-Critic:** Combine both! Used in modern systems like ChatGPT's RLHF training
- Try more Gymnasium environments: `CartPole-v1`, `MountainCar-v0`, `LunarLander-v3`